# Limits and Continuity

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 00.01–00.04, 01.01–01.10 |
| Time | ~50 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/01_limits_and_continuity.ipynb)

---

## 🎯 Learning Objective

Build a rigorous notion of what it means for a function to approach a value, and connect continuity to why certain activation functions behave well during training.

---

## 📐 Theory

Calculus begins with a question that sounds trivial and isn't: what does it mean for $f(x)$ to
*approach* a value? "Get close to" is intuitive until it must be precise enough to prove
theorems with — which is what the $\epsilon$-$\delta$ definition does.

### The $\epsilon$-$\delta$ definition

We say $\lim_{x \to a} f(x) = L$ if, for every tolerance $\epsilon > 0$ on the output, there is
a corresponding tolerance $\delta > 0$ on the input such that:

$$0 < |x - a| < \delta \implies |f(x) - L| < \epsilon$$

Read it as a game: you pick how close to $L$ you want $f(x)$ to be (your $\epsilon$); I must
produce a $\delta$-window around $a$ guaranteeing that for every $x$ in it. If I can always
win no matter how small your $\epsilon$, the limit exists and equals $L$. This flips the usual
direction of reasoning — from a target *output* precision, work backward to the required
*input* precision — the same flow used later for numerical stability (Module 06) and gradient
behavior (starting `02.02`).

### Limit laws

Provided the individual limits exist, limits pass through arithmetic:

$$\lim_{x\to a}[f(x) + g(x)] = \lim_{x\to a} f(x) + \lim_{x\to a} g(x), \qquad
\lim_{x\to a}[f(x)\cdot g(x)] = \Big(\lim_{x\to a} f(x)\Big)\Big(\lim_{x\to a} g(x)\Big)$$

and similarly for quotients (denominator's limit $\ne 0$). This is why you can compute the
limit of a complicated expression piece by piece instead of always returning to $\epsilon$-$\delta$.

### Continuity

$f$ is **continuous at $a$** if $f(a)$ is defined, $\lim_{x\to a} f(x)$ exists, and the two
match — no jumps, holes, or blow-ups. Discontinuities differ in kind: **removable** (the limit
exists but disagrees with $f(a)$ — patch one point and it's fixed), **jump** (left- and
right-hand limits both exist but disagree — no patch fixes this), and **infinite/essential**
(the function blows up or oscillates without settling). Sums, products, and compositions of
continuous functions are continuous wherever defined, so a handful of "atomic" continuous
functions ($\sin$, $e^x$, polynomials) generate a huge family for free.

### The Intermediate Value Theorem (IVT)

If $f$ is continuous on $[a,b]$ and $L$ lies between $f(a)$ and $f(b)$, some $c \in [a,b]$ has
$f(c)=L$: a continuous function cannot "jump over" a value. This is false for discontinuous
functions (a step function can leap from $0$ to $1$ without ever equaling $0.5$), and it's the
backbone of bisection-style root-finding, used below.

Every derivative from `02.02` onward is *defined* as a limit, so this isn't a historical
detour — it's the ground floor. Continuity also matters practically: gradient-based learning
needs the function it optimizes to behave predictably as inputs shift by a tiny amount, made
concrete below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Draw a horizontal band of width $2\epsilon$ around $L$, then ask how wide a vertical band
around $a$ keeps the function trapped inside it. For a continuous function you can always
find such a band; for a jump, no band works once $\epsilon$ is smaller than half the jump.

In [ ]:
# The epsilon-delta "trapping band" game, continuous vs. discontinuous
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: f(x)=x^2 at a=1, L=1 -- a delta-band around a keeps f(x) inside the epsilon-band
a, L, epsilon = 1.0, 1.0, 0.3
delta = min(abs(np.sqrt(L + epsilon) - a), abs(a - np.sqrt(L - epsilon)))
x = np.linspace(0, 2, 400)
axes[0].plot(x, x ** 2, color="#4C72B0", lw=2, label=r"$f(x)=x^2$")
axes[0].axhspan(L - epsilon, L + epsilon, color="#DD8452", alpha=0.25, label=r"$L \pm \epsilon$")
axes[0].axvspan(a - delta, a + delta, color="#55A868", alpha=0.2, label=r"$a \pm \delta$")
axes[0].plot(a, L, 'ko', zorder=5)
axes[0].set_xlim(0, 2); axes[0].set_ylim(0, 4); axes[0].set_xlabel("x"); axes[0].set_ylabel("f(x)")
axes[0].set_title(f"Continuous: delta={delta:.2f} traps f(x)")
axes[0].legend(loc="upper left")

# Right: a jump discontinuity -- once epsilon < half the jump size, no delta works
x2 = np.linspace(0, 2, 2000)
f2 = np.floor(x2 * 2) / 2
axes[1].plot(x2, f2, color="#4C72B0", lw=2, label=r"$f(x)=\lfloor 2x \rfloor/2$")
jump_val = f2[np.argmin(np.abs(x2 - 1.0))]
axes[1].axhspan(jump_val - 0.15, jump_val + 0.15, color="#DD8452", alpha=0.25, label=r"$L \pm \epsilon$ (too narrow)")
axes[1].axvline(1.0, color="gray", ls="--", lw=1)
axes[1].set_xlim(0, 2); axes[1].set_ylim(-0.2, 2.2); axes[1].set_xlabel("x"); axes[1].set_ylabel("f(x)")
axes[1].set_title("Discontinuous: no delta works\nonce epsilon < half the jump")
axes[1].legend(loc="upper left")
plt.tight_layout()
plt.show()

print(f"Continuous case: largest valid delta for epsilon={epsilon} is ~{delta:.4f}")
print("Discontinuous case: shrinking epsilon below 0.25 makes NO delta work -- the limit doesn't exist at the jump.")

## 🐍 Implementation from Scratch

We implement the $\epsilon$-$\delta$ check as a search procedure so the definition stops being
abstract, then implement bisection, which turns the IVT into a root-finding algorithm.

In [ ]:
def find_valid_delta(f, a, L, epsilon, delta_start=1.0, shrink=0.5, n_tries=200):
    """Search for a delta with 0 < |x-a| < delta => |f(x)-L| < epsilon, by shrinking
    delta on a fine grid -- a numerical stand-in for the algebraic epsilon-delta proof."""
    delta = delta_start
    for _ in range(n_tries):
        pts = np.linspace(a - delta, a + delta, 401)
        pts = pts[np.abs(pts - a) > 1e-12]  # exclude x = a itself
        if np.all(np.abs(f(pts) - L) < epsilon):
            return delta
        delta *= shrink
    return None

def bisection_root(f, lo, hi, tol=1e-8, max_iter=100):
    """Find a root via the IVT: opposite-signed f(lo), f(hi) plus continuity guarantee
    a root between them, so we keep halving toward it."""
    f_lo, f_hi = f(lo), f(hi)
    assert f_lo * f_hi < 0, "f(lo), f(hi) must have opposite signs for IVT to apply"
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        f_mid = f(mid)
        if abs(f_mid) < tol:
            return mid
        if f_lo * f_mid < 0:  # root is in the half still bracketing a sign change
            hi, f_hi = mid, f_mid
        else:
            lo, f_lo = mid, f_mid
    return (lo + hi) / 2

f_sq = lambda x: x ** 2
for eps in [0.5, 0.1, 0.01, 0.001]:
    print(f"epsilon={eps:<8} -> delta ~ {find_valid_delta(f_sq, 1.0, 1.0, eps):.6f}")

f_cubic = lambda x: x ** 3 - x - 2  # root near x=1.5214, doesn't factor nicely
root = bisection_root(f_cubic, lo=1.0, hi=2.0)
print(f"\nBisection root of x^3-x-2 on [1,2]: {root:.6f}  (f(root)={f_cubic(root):.2e})")
print("Bisection relies ENTIRELY on continuity + IVT -- invalid on a discontinuous function.")

## 🤖 Why This Matters for AI

Early neural networks used the **step function** ($f(x)=1$ if $x>0$ else $0$) as an activation
— fire or don't, like a biological neuron. But it's discontinuous at $0$ and its slope is $0$
everywhere it's defined. Gradient descent needs $\partial\text{loss}/\partial w$ to be a
useful, non-zero signal; with a step activation that signal vanishes almost everywhere, so
weights never update no matter how wrong the network is. This is why the field moved to
**continuous** activations (sigmoid, tanh, later ReLU): continuity is the minimum bar for a
derivative to exist almost everywhere, which is the minimum bar for backprop to carry any
signal. Below, a single neuron with each activation learns a toy task — the step version gets
permanently stuck.

In [ ]:
# Discontinuous vs. continuous activations: why gradient descent needs continuity to work
rng = np.random.default_rng(0)
X = rng.uniform(-2, 2, size=(200, 2))
y = (X[:, 0] + X[:, 1] > 0).astype(float)  # linearly separable toy task

def train_neuron(use_step, n_steps=300, lr=0.5):
    w, b = rng.normal(scale=0.1, size=2), 0.0
    losses = []
    for _ in range(n_steps):
        z = X @ w + b
        if use_step:
            pred = (z > 0).astype(float)
            # step'(z) = 0 everywhere it's defined -- use the TRUE derivative, not a
            # heuristic substitute like the perceptron rule, to show what calculus alone gives you
            dz = (pred - y) * 0.0
        else:
            pred = 1.0 / (1.0 + np.exp(-z))
            dz = (pred - y) * pred * (1 - pred)  # sigmoid'(z), nonzero almost everywhere
        w -= lr * (X.T @ dz) / len(y)
        b -= lr * np.mean(dz)
        losses.append(np.mean((pred - y) ** 2))
    return np.array(losses), w

step_losses, w_step = train_neuron(use_step=True)
sigmoid_losses, w_sig = train_neuron(use_step=False)
print(f"Step activation:    loss {step_losses[0]:.4f} -> {step_losses[-1]:.4f}  (barely moves)")
print(f"Sigmoid activation: loss {sigmoid_losses[0]:.4f} -> {sigmoid_losses[-1]:.4f}  (converges)")
print(f"Sigmoid weights approach the true separator direction (1,1): {np.round(w_sig, 3)}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(step_losses, color="#C44E52", lw=2, label="step activation (discontinuous)")
ax.plot(sigmoid_losses, color="#4C72B0", lw=2, label="sigmoid activation (continuous)")
ax.set_xlabel("training step"); ax.set_ylabel("mean squared error")
ax.set_title("Continuity is what lets gradient descent make progress at all")
ax.legend()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $f(x) = 3x + 1$, $a = 2$, $L = 7$, $\epsilon = 0.3$, solve algebraically for the largest
   $\delta$ satisfying the $\epsilon$-$\delta$ definition, then verify with `find_valid_delta`.
   As a sanity check, confirm your $\delta$ is tight in both directions: the definition should
   hold at your $\delta$ and fail for some $\delta' > \delta$ (try $\delta'=0.11$).

<details>
<summary>💡 Solution</summary>

$|f(x)-L| = |3x+1-7| = 3|x-2| < \epsilon = 0.3 \iff |x-2| < 0.1$, so the largest valid
$\delta$ is exactly $0.1$. Checking both directions confirms tightness: at $\delta=0.1$ every
$x$ with $0<|x-2|<0.1$ satisfies $|f(x)-7|<0.3$, but at $\delta'=0.11$ it fails — e.g.
$x=2.105$ has $|x-2|=0.105<0.11$ yet $|f(x)-7| = 3(0.105) = 0.315 > 0.3$. `find_valid_delta`
starts from $\delta=1.0$ and only halves, so it never lands on $0.1$ exactly — it reports
$0.0625$ (the largest power-of-two-shrunk value below $0.1$), a valid if not the tightest
$\delta$, consistent with the algebraic bound.

</details>

### 💭 UNDERSTAND
2. `bisection_root(lambda x: 1/x, lo=-1.0, hi=4.0)` runs to completion with no error and
   returns a value close to $0$, even though $f(x)=1/x$ has no root anywhere ($1/x$ is never
   $0$). Explain which specific hypothesis of the IVT this call violates, and why the function
   silently returns a wrong-looking answer instead of raising an error.

<details>
<summary>💡 Solution</summary>

Bisection's correctness depends entirely on $f$ being **continuous** on the bracketing
interval $[\text{lo},\text{hi}]$ — that's the hypothesis the IVT needs to guarantee a root
exists between two opposite-signed endpoints. Here $f(x)=1/x$ is discontinuous (undefined,
blowing up) at $x=0$, which lies inside $(-1,4)$, so the hypothesis is violated even though the
*sign condition* the code explicitly checks ($f(\text{lo})\cdot f(\text{hi})<0$) happens to
hold. The algorithm has no way to detect this — it only ever evaluates $f$ at a few points and
checks signs, and near $x=0$ the sign of $1/x$ really does flip (from very negative to very
positive) exactly as it would around a genuine root, which is what fools bisection into
repeatedly narrowing the bracket toward $x=0$ and eventually reporting a tiny value near it as
if it were a root, with no mechanism to notice that $f$ is exploding rather than crossing zero.

</details>

### ✏️ DERIVE
3. For $f(x) = x^2$ at an arbitrary point $a$ (any sign) with tolerance $\epsilon > 0$, derive
   a valid $\delta$ using the standard bounding trick: first restrict to $\delta \le 1$ so that
   $|x-a|<\delta$ forces $|x+a| \le 2|a|+1$, then bound $|f(x)-a^2| = |x-a||x+a|$. State the
   final formula for $\delta$ in terms of $a$ and $\epsilon$.

<details>
<summary>💡 Solution outline</summary>

Write $|f(x)-a^2| = |x^2-a^2| = |x-a|\cdot|x+a|$. Additionally requiring $\delta \le 1$ means
$|x-a|<\delta\le 1$, so $|x+a| = |(x-a)+2a| \le |x-a|+2|a| < 1+2|a|$, giving
$|x-a||x+a| < \delta(2|a|+1)$. Forcing this below $\epsilon$ requires
$\delta<\epsilon/(2|a|+1)$. Combining both requirements gives the standard construction
$$\delta = \min\!\left(1, \frac{\epsilon}{2|a|+1}\right).$$
This $\delta$ is *sufficient* but not always the *tightest* possible — e.g. at $a=1,
\epsilon=0.3$ it gives $\delta=0.1$, while the exact tightest boundary (solving
$1-\delta=\sqrt{1-\epsilon}$ from the notebook's own Visual Intuition cell) is closer to
$0.140$ — both are valid $\delta$'s satisfying the definition, since the definition only
requires *some* working $\delta$ to exist for each $\epsilon$, not the largest one.

</details>

### 🐍 IMPLEMENT
4. Write `classify_discontinuity(f, a)` that samples $f$ at $a\pm h$ and $a\pm h/10$ for small
   $h$ and returns one of `"continuous"`, `"removable"`, `"jump"`, or `"infinite/essential"`.
   Test it on $f(x) = \frac{\sin x}{x}$ (defined as $1$ at $x=0$), $g(x)=\frac{1}{x}$, and
   $h(x)=\text{sign}(x)$, each at $a=0$, and check every classification against SymPy's
   one-sided `sp.limit(..., '-')` / `sp.limit(..., '+')`.

<details>
<summary>✅ How to know you're right</summary>

Your function should report $f$ as `"continuous"` (SymPy: both one-sided limits equal $1$,
matching the patched $f(0)=1$), $g$ as `"infinite/essential"` (SymPy: limits are $-\infty$ and
$+\infty$), and $h$ as `"jump"` (SymPy: limits are $-1$ and $1$ — disagreeing, but both
finite). As an additional sanity check, run it on a function you're certain is continuous
everywhere (e.g. $\sin(x)$, at any point) and confirm it never gets misclassified as one of the
other three categories — a classifier that can't correctly recognize the "boring" continuous
case isn't trustworthy on the interesting ones either.

</details>

### 🤖 APPLY
5. The step activation's *true* derivative is $0$ everywhere it's defined, which is why
   `train_neuron(use_step=True)` never updates its weights at all. A common workaround in
   binarized/quantized networks is the **straight-through estimator (STE)**: keep the true step
   function on the forward pass, but substitute a fake, nonzero "gradient" on the backward pass
   (e.g. pretend the activation was the identity, so `dz = (pred - y) * 1`). Implement this and
   train the same toy task.

<details>
<summary>✅ What you should observe</summary>

Unlike the true-gradient step neuron (loss stuck at exactly its initial value, weights
literally unchanged after 300 steps), the STE version's loss should drop substantially — in
one run, from $0.52$ to $0.0$ — and its learned weight direction should align closely with the
true separator direction $(1,1)/\sqrt2$ (cosine similarity above $0.999$), reaching perfect
classification accuracy on the toy task. This is a real technique, not a toy simplification:
whenever a forward pass needs a genuinely discontinuous or non-differentiable operation
(binarization, quantization, top-$k$ selection), some substitute gradient is required for
backprop to carry any signal at all — the true derivative being zero (or undefined) almost
everywhere is exactly the failure mode this notebook's own step-vs-sigmoid comparison
demonstrates, and STE is one standard way around it, at the cost of the backward pass no longer
computing the forward pass's actual derivative.

</details>

### 🚀 CHALLENGE
6. The GELU activation (used in GPT-style transformers) is $\text{GELU}(x) = x\cdot\Phi(x)$,
   $\Phi$ the standard normal CDF. (a) Plot GELU alongside ReLU and the step function.
   (b) At $x_0=0$ (ReLU's kink), compute the central second difference
   $\frac{f(x_0+h)-2f(x_0)+f(x_0-h)}{h^2}$ for both ReLU and GELU across several shrinking $h$,
   and characterize how each behaves as $h\to0$. (c) Connect what you find to why the
   second-order optimization methods in Module `04.07` (which use curvature, i.e. second
   derivatives) can behave qualitatively differently on GELU networks than on ReLU networks.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports that ReLU's central second difference at its kink does not converge to
anything as $h\to0$ — it scales exactly like $1/h$ (e.g. $10, 100, 1000$ as $h$ shrinks through
$0.1, 0.01, 0.001$), diverging, because ReLU's first derivative has a genuine jump there and a
second derivative doesn't exist even in an extended sense. GELU's central second difference, by
contrast, *converges* as $h\to0$, to $\sqrt{2/\pi}\approx0.7979$, with error shrinking by about
$4\times$ each time $h$ is halved — the signature of a well-defined, $O(h^2)$-accurate second
derivative. A strong answer connects this to Module `04.07`: methods that use curvature
(Newton's method, quasi-Newton methods) implicitly assume the loss surface has a meaningful,
finite second derivative to exploit; on a ReLU network the loss surface has kinks almost
everywhere a unit switches on or off, so "curvature" is either undefined or concentrated in
Dirac-delta-like spikes at those switching boundaries, whereas GELU's smoothness means
curvature information is genuinely available everywhere — part of why smoother activations are
sometimes preferred when a training method leans on second-order information, even though
first-order methods (plain SGD/Adam) tolerate ReLU's kinks just fine in practice.

</details>

---

## 📚 Further Reading
- Spivak, *Calculus*, Ch. 5–6 (Limits and Continuity)
- Nielsen, [*Neural Networks and Deep Learning*, Ch. 4](http://neuralnetworksanddeeplearning.com/chap4.html)
- Hendrycks & Gimpel, ["Gaussian Error Linear Units (GELUs)"](https://arxiv.org/abs/1606.08415)

---

*Next notebook: [Derivatives and Differentiation Rules](02_derivatives_and_rules.ipynb)*